# Teleport: chess positions -> Gazebo piece poses (library)

Pieces in the Gazebo world are static models moved only by the `set_pose` service ("vanishing
pieces"): a move = teleport the captured piece to the graveyard, then the moving piece to its
target square. This notebook keeps track of *which model instance* (`w_P_5`, `b_Q_2`, ...) stands
on which square, turns moves / FENs / whole PGN games into `set_pose` requests, and has backends
that send them (gz CLI, ROS 2 service).

**Deliverables** (README "Updated plan (11/4/2026)"): *PGN teleport replay script*, and
*Gazebo vanishing pieces to test game logic* (compare Gambit's moves to ground truth, verify
replayed games follow the rules: python-chess does the legality, we only move models).

**TODO checklist**
- [ ] instance assignment for the start position and for any FEN (spare queens, graveyard)
- [ ] move -> requests: captures, en passant, castling, promotion (spare queen / reuse captured)
- [ ] PGN replay -> per-ply request lists
- [ ] request formatting for `gz service` and `ros2 service call`
- [ ] `Ros2Backend.send` with rclpy (persistent node + client, much faster than the CLI)
- [ ] underpromotion when no captured piece of that type is available (add spare N/B/R models?)
- [ ] optional: animate (lift, move, drop) instead of an instant teleport, for nicer videos

In [ ]:
%run ./world_generator.ipynb

In [ ]:
import shutil
import subprocess
from dataclasses import dataclass, field

import chess
import chess.pgn

## A set_pose request

Gazebo's `/world/<world>/set_pose` takes a `gz.msgs.Pose` (name + position + orientation) and
returns `gz.msgs.Boolean`. Bridged to ROS 2 it is `ros_gz_interfaces/srv/SetEntityPose` with
`entity: {name, type: 2 (MODEL)}`. Orientation is a quaternion (x, y, z, w); identity = upright.

In [ ]:
@dataclass
class SetPose:
    name: str
    position: tuple                     # (x, y, z) meters, world frame
    orientation: tuple = (0.0, 0.0, 0.0, 1.0)   # quaternion (x, y, z, w)

    def gz_req(self):
        """gz.msgs.Pose request text: name, position {x y z}, orientation {x y z w}."""
        # Design: protobuf text format (the `--req` argument of `gz service`); 4 decimals for
        # position, more for the quaternion. Quote the name.
        raise NotImplementedError

    def gz_command(self, world=WORLD_NAME, timeout_ms=2000):
        """argv list for subprocess: gz service -s /world/<world>/set_pose ... --req <gz_req()>."""
        # Design: reqtype gz.msgs.Pose, reptype gz.msgs.Boolean, --timeout in ms. A list (not a
        # shell string) avoids quoting problems with the braces in the request.
        raise NotImplementedError

    def ros2_request(self):
        """Dict matching ros_gz_interfaces/srv/SetEntityPose.Request (entity type 2 = MODEL)."""
        # Design: nested dict {entity: {name, type}, pose: {position: {x,y,z}, orientation: {x,y,z,w}}}.
        raise NotImplementedError

    def ros2_command(self, world=WORLD_NAME):
        """argv list for `ros2 service call /world/<world>/set_pose ros_gz_interfaces/srv/SetEntityPose <yaml>`."""
        # Design: json.dumps(ros2_request()) is valid YAML, so it can be the request argument.
        raise NotImplementedError

def color_char(color):
    # Design: chess.WHITE -> 'w', chess.BLACK -> 'b' (the instance-name prefix).
    raise NotImplementedError

## Piece tracker

The tracker maps squares to instance names, and keeps the graveyard (slots along +x; the spare
queens start in slots 0..3, captured pieces take the lowest free slot).

Rules for promotions: first reuse a *captured* piece of the promoted type and color (sitting in
the graveyard), otherwise for a queen take an unused spare queen. Anything else is a TODO.

`apply_move(move)` returns the requests **and** pushes the move onto the internal board, so the
tracker always mirrors the game. Captured piece goes first so two models never overlap.

In [ ]:
@dataclass
class PieceTracker:
    board: chess.Board = field(default_factory=chess.Board)
    on_square: dict = field(default_factory=dict)     # chess square index -> instance name
    in_grave: dict = field(default_factory=dict)      # instance name -> graveyard slot

    @staticmethod
    def instance_kind(name):
        """'w_Q_2' -> (chess.WHITE, chess.QUEEN)."""
        # Design: split the name on '_'; color from the prefix, piece type via chess.Piece.from_symbol.
        raise NotImplementedError

    @classmethod
    def from_fen(cls, fen=chess.STARTING_FEN, cfg=BOARD):
        """Assign model instances to the pieces of any position.

        Pieces on their start square get their start instance (so the start FEN reproduces the
        world exactly); the rest take free instances of the same color/kind in name order; unused
        instances go to the graveyard. Raises ValueError when there are not enough models
        (e.g. 3 knights of one color).
        """
        # Design:
        # - Pool of instances = start_instances(cfg); remember each one's start square.
        # - Pass 1: a piece standing on the start square of a free instance of the same color/kind
        #   takes that instance (this makes the start FEN reproduce the repo world exactly).
        # - Pass 2: remaining pieces take any free instance of the right color/kind, deterministic
        #   order; none left -> ValueError naming the square.
        # - Everything still free goes to consecutive graveyard slots.
        # Why two passes: one pass in square order could steal e.g. w_R_2 for a rook that is not on h1.
        raise NotImplementedError

    # -- graveyard -------------------------------------------------------------------------
    def _free_slot(self):
        # Design: lowest graveyard slot index not used in in_grave.
        raise NotImplementedError

    def _bury(self, name, cfg=BOARD):
        # Design: put `name` in the lowest free slot (update in_grave) and return the SetPose there.
        raise NotImplementedError

    def _take_for_promotion(self, color, piece_type):
        # Design:
        # - Look in the graveyard for an instance of that color and piece type; take one (decide
        #   which: lowest slot keeps the row tidy and uses the spare queens first) and remove it
        #   from in_grave.
        # - None available (underpromotion to N/B/R with nothing captured): NotImplementedError for
        #   now; options: add spare N/B/R models to the world, or fall back to a queen model.
        raise NotImplementedError

    # -- moves -----------------------------------------------------------------------------
    def poses(self, cfg=BOARD):
        """Current {instance: (x, y, z)} for every instance (board + graveyard)."""
        # Design: square_center for on_square entries, graveyard_pose for in_grave entries.
        raise NotImplementedError

    def requests_for_all(self, cfg=BOARD):
        """Requests that put every instance where the tracker says (use after from_fen / reset)."""
        # Design: one SetPose per instance from poses(); sort by name so the order is reproducible.
        raise NotImplementedError

    def apply_move(self, move, cfg=BOARD):
        """Push `move` (chess.Move or UCI/SAN string) and return the list of SetPose requests."""
        # Design:
        # - Parse strings: try UCI first (chess.Move.from_uci), fall back to SAN (board.parse_san).
        #   Reject illegal moves (not in board.legal_moves) with ValueError.
        # - Order of requests: captured piece to the graveyard FIRST (so two models never overlap),
        #   then the mover, then (castling) the rook.
        # - En passant (board.is_en_passant): the captured pawn is one rank behind the target square.
        # - Promotion: the pawn is buried and a model of the promoted type (_take_for_promotion)
        #   appears on the target square.
        # - Castling (board.is_castling / is_kingside_castling): also move the rook h->f or a->d.
        # - Update on_square for every model that moved, and push the move LAST (the checks above
        #   need the board before the move).
        raise NotImplementedError

    def check_consistent(self):
        """The tracker's squares/kinds must match python-chess's board exactly."""
        # Design: same occupied squares as board.piece_map(), each instance's kind matches the
        # piece on its square, and no instance is both on the board and in the graveyard.
        # Assert (or raise) with a helpful message; return True when all good.
        raise NotImplementedError

## PGN replay

One entry per ply: index (1 = first move; 0 is the start position with all poses), SAN, UCI, FEN
after the move, requests. `run_replay_game.ipynb` sends these with a backend and (optionally)
grabs a camera frame after each ply.

In [ ]:
@dataclass
class Ply:
    idx: int
    san: str
    uci: str
    fen: str
    requests: list

def pgn_replay(pgn_text, cfg=BOARD):
    """PGN text (first game) -> [Ply]. Ply 0 resets every model to the game's start position."""
    # Design:
    # - chess.pgn.read_game on a StringIO; None -> ValueError.
    # - Tracker from the game's start board (handles PGNs with a FEN header).
    # - Ply 0 = all poses (requests_for_all); then for each mainline move: SAN (computed BEFORE
    #   pushing), apply_move requests, FEN after the move.
    raise NotImplementedError

def read_pgn_games(path):
    """Yield chess.pgn.Game objects from a (multi-game) PGN file."""
    # Design: generator; call chess.pgn.read_game repeatedly on the open file until it returns None.
    raise NotImplementedError

## Backends

* `DryRunBackend` records requests (tests, notebooks without Gazebo).
* `GzCliBackend` shells out to `gz service` (simple, ~50 ms per call; fine for replays).
  Needs Gazebo on PATH; the world must be running.
* `Ros2Backend` uses the bridged ROS 2 service (start the bridge from docs/gazebo.md step 4,
  or `gambit.launch.py`, which starts it). TODO.

In [ ]:
class DryRunBackend:
    def __init__(self):
        self.sent = []
    def send(self, req):
        # Design: record the request in self.sent; return True (success).
        raise NotImplementedError
    def send_all(self, reqs):
        # Design: send each request in order; True only if all succeeded (stop at the first failure?).
        raise NotImplementedError

class GzCliBackend(DryRunBackend):
    def __init__(self, world=WORLD_NAME, timeout_ms=2000):
        # Design: fail early with a clear RuntimeError if `gz` is not on PATH (shutil.which);
        # store world and timeout; keep the `sent` list from DryRunBackend.
        raise NotImplementedError
    def send(self, req):
        # Design:
        # - subprocess.run(req.gz_command(...), capture_output=True, text=True).
        # - Success = exit code 0 AND the reply says "true" (gz prints data: true); otherwise raise
        #   with stdout/stderr so a typo in a model name is obvious.
        raise NotImplementedError

class Ros2Backend(DryRunBackend):
    """TODO: rclpy client for /world/<world>/set_pose (ros_gz_interfaces/srv/SetEntityPose).

    Hints: try: import rclpy; from ros_gz_interfaces.srv import SetEntityPose; from
    ros_gz_interfaces.msg import Entity. Create one node + client in __init__, wait_for_service,
    then for each request fill req.entity.name / req.entity.type = Entity.MODEL / req.pose and
    rclpy.spin_until_future_complete(node, client.call_async(req), timeout_sec=2.0).
    See ros_playground.ipynb for a sketch.
    """
    def __init__(self, world=WORLD_NAME):
        # Design: import rclpy lazily (clear RuntimeError if missing), one node + one client, wait_for_service.
        raise NotImplementedError("TODO: Ros2Backend")

## Self-check

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - PieceTracker.from_fen() (start FEN): poses() == start_poses() and check_consistent()
# - docs/gazebo.md: apply_move('e2e4') -> exactly one request, w_P_5 to (0.2025, -0.0125, 0.006);
#   its gz_req() contains 'name: "w_P_5"' and ros2_request() has entity type 2
# - capture: 1.e4 d5 2.exd5 -> requests [b_P_4, w_P_5], b_P_4 goes to graveyard slot 4
#   (slots 0..3 hold the spare queens)
# - castling: FEN 'r3k2r/8/8/8/8/8/8/R3K2R w KQkq - 0 1', e1g1 -> [w_K_1, w_R_2], rook to f1
# - en passant: e4 a6 e5 d5, then e5d6 -> first request buries b_P_4, mover lands on d6
# - promotion: '8/P7/8/8/8/8/8/k6K w - - 0 1', a7a8q -> w_P_1 buried, a w_Q_* lands on a8, consistent
# - pgn_replay(Scholar's mate) -> 8 plies, ply 0 has 36 requests, last SAN 'Qxf7#',
#   last requests [b_P_6, w_Q_1]; DryRunBackend.send_all records them
# - not enough models: from_fen('NNN5/8/8/8/8/8/8/k6K w - - 0 1') raises ValueError
# Example shape of a test:
# t = PieceTracker.from_fen(); r = t.apply_move("e2e4"); assert [q.name for q in r] == ["w_P_5"]
print("Self-check: not written yet (see plan above)")